# Demo 08 — External OIDC (Keycloak tenant-a)

Flow matches upstream MaaS samples / e2e:

1. Password-grant against Keycloak **`tenant-a`** / client **`test-client`**
2. Mint MaaS API key on the **oidc** tenant gateway with the OIDC bearer
3. List models + BBR chat

**Upstream:** [docs/samples/install/keycloak](https://github.com/opendatahub-io/models-as-a-service/tree/main/docs/samples/install/keycloak), [external-oidc.md](https://github.com/opendatahub-io/models-as-a-service/blob/main/docs/content/advanced-administration/external-oidc.md)

**Prereq:** `./demos/08-maas-35-features/scripts/apply-oidc-tenant.sh` (runs upstream `setup-keycloak.sh` + `apply-test-realms.sh`).

Test user **`alice_lead`** / **`letmein`** → groups `Engineering`, `Project-Alpha` (hardcoded in upstream test realms — lab only).


## Demo quick swap


In [ ]:
DEMO_OIDC_BASE = ""          # https://oidc-maas.<domain>
DEMO_KEYCLOAK_HOST = ""      # keycloak.<domain> (no https://)
DEMO_OIDC_USERNAME = "alice_lead"
DEMO_OIDC_PASSWORD = "letmein"
DEMO_OIDC_CLIENT_ID = "test-client"
DEMO_OIDC_REALM = "tenant-a"


## Setup


In [ ]:
import base64
import json
import os
import ssl
import urllib.error
import urllib.parse
import urllib.request
from typing import Any, Dict, Optional

def _g(name, env="", default=""):
    v = globals().get(name, "")
    if isinstance(v, str) and v.strip():
        return v.strip()
    return (os.environ.get(env) or default).strip()

OIDC_BASE = _g("DEMO_OIDC_BASE", "MAAS_OIDC_BASE", "https://oidc-maas.YOUR_DOMAIN_HERE").rstrip("/")
KEYCLOAK_HOST = _g("DEMO_KEYCLOAK_HOST", "KEYCLOAK_HOST", "keycloak.YOUR_DOMAIN_HERE")
if KEYCLOAK_HOST.startswith("https://"):
    KEYCLOAK_HOST = KEYCLOAK_HOST[len("https://"):]
USERNAME = _g("DEMO_OIDC_USERNAME", "OIDC_USERNAME", "alice_lead")
PASSWORD = _g("DEMO_OIDC_PASSWORD", "OIDC_PASSWORD", "letmein")
CLIENT_ID = _g("DEMO_OIDC_CLIENT_ID", "OIDC_CLIENT_ID", "test-client")
REALM = _g("DEMO_OIDC_REALM", "OIDC_REALM", "tenant-a")
VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")
TOKEN_URL = f"https://{KEYCLOAK_HOST}/realms/{REALM}/protocol/openid-connect/token"
ISSUER = f"https://{KEYCLOAK_HOST}/realms/{REALM}"


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def http_json(method: str, url: str, *, token: Optional[str] = None, data: Optional[Dict[str, Any]] = None,
              form: Optional[Dict[str, str]] = None):
    headers = {"Accept": "application/json"}
    body = None
    if form is not None:
        body = urllib.parse.urlencode(form).encode("utf-8")
        headers["Content-Type"] = "application/x-www-form-urlencoded"
    elif data is not None:
        body = json.dumps(data).encode("utf-8")
        headers["Content-Type"] = "application/json"
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=_ssl_ctx(), timeout=120) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err) if err else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


def decode_jwt_payload(token: str) -> dict:
    part = token.split(".")[1]
    pad = "=" * (-len(part) % 4)
    return json.loads(base64.urlsafe_b64decode(part + pad))


print("OIDC_BASE   :", OIDC_BASE)
print("TOKEN_URL   :", TOKEN_URL)
print("ISSUER      :", ISSUER)
print("CLIENT_ID   :", CLIENT_ID)
print("USER        :", USERNAME)


## 1. Keycloak password grant → access token


In [ ]:
_, tok = http_json(
    "POST",
    TOKEN_URL,
    form={
        "grant_type": "password",
        "client_id": CLIENT_ID,
        "username": USERNAME,
        "password": PASSWORD,
    },
)
OIDC_TOKEN = tok.get("access_token")
if not OIDC_TOKEN:
    raise RuntimeError(f"No access_token: {tok}")

payload = decode_jwt_payload(OIDC_TOKEN)
print("OIDC token prefix:", OIDC_TOKEN[:24] + "…")
print("azp / client:", payload.get("azp") or payload.get("aud"))
print("groups claim:", payload.get("groups"))
print("preferred_username:", payload.get("preferred_username"))


## 2. Mint MaaS API key with OIDC bearer

`POST {oidc-gateway}/maas-api/v1/api-keys` — same path as upstream e2e / [API key management](https://github.com/opendatahub-io/models-as-a-service/blob/main/docs/content/user-guide/api-key-management.md).


In [ ]:
_, key_body = http_json(
    "POST",
    f"{OIDC_BASE}/maas-api/v1/api-keys",
    token=OIDC_TOKEN,
    data={
        "name": "demo08-oidc-notebook",
        "description": "External OIDC workbook",
        "expiresIn": "24h",
        "subscription": "demo08-oidc-catalog",
    },
)
API_KEY = key_body.get("key")
if not API_KEY:
    raise RuntimeError(f"No key in response: {key_body}")
print("API key prefix:", API_KEY[:20] + "…")


## 3. Discover models + BBR chat on oidc tenant


In [ ]:
_, models = http_json("GET", f"{OIDC_BASE}/maas-api/v1/models", token=API_KEY)
data = models.get("data") or []
if not data:
    raise SystemExit("Empty catalog — check demo08-oidc-catalog / Engineering groups on the token.")

print("OIDC tenant catalog:")
for m in data:
    print(" -", m.get("id") or m.get("name"), "→", m.get("url"))

mid = data[0].get("id") or data[0].get("name")
status, completion = http_json(
    "POST",
    f"{OIDC_BASE}/v1/chat/completions",
    token=API_KEY,
    data={
        "model": mid,
        "messages": [{"role": "user", "content": "Say hello from the OIDC tenant."}],
        "max_tokens": 48,
    },
)
choices = completion.get("choices") or []
msg = (choices[0].get("message") or {}) if choices and isinstance(choices[0], dict) else {}
print()
print("HTTP", status, "| model:", mid)
print("Assistant:", msg.get("content") or "(empty)")
